In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
DEVICE = "mps" if torch.mps.is_available() else "cpu" 

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

mps


In [7]:
res_net = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1).to(DEVICE)

In [8]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

In [9]:
class ImageDataset(Dataset):
    def __init__(self, root_dir, df):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        w, h = img.size if isinstance(img, Image.Image) else (img.shape[2], img.shape[1])
        min_side = min(w, h)
        img_cropped = transforms.CenterCrop(min_side)(img)
        img_resized = transforms.Resize((256, 256))(img_cropped)
        img_tensor = transforms.ToTensor()(img_resized)
        
        return img_tensor

In [10]:
tr_images_ds = ImageDataset(IMAGE_FOLDER, train_df)

batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [11]:
sureness = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure = res_net(batch.to(DEVICE))

    sure = image_sure.to('cpu').numpy()

    sureness.extend(sure)

100%|██████████| 1065/1065 [04:51<00:00,  3.66it/s]


In [12]:
DATA_PATH = "/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data"

train_text_emb = np.load(f'{DATA_PATH}/train_text_embeddings.npy', allow_pickle=True).tolist()
train_img_emb = np.load(f'{DATA_PATH}/train_img_embedings.npy', allow_pickle=True).tolist()

test_text_emb = np.load(f'{DATA_PATH}/test_text_embeddings.npy', allow_pickle=True).tolist()
test_img_emb = np.load(f'{DATA_PATH}/test_img_embedings.npy', allow_pickle=True).tolist()

In [13]:
for i in range(len(train_text_emb[-1])):
    train_df[f"text{i}"] = [train_text_emb[j][i] for j in range(len(train_text_emb))]

for i in range(len(train_img_emb[-1])):
    train_df[f"img{i}"] = [train_img_emb[j][i] for j in range(len(train_img_emb))]

for i in range(1000):
    train_df[f"f{i}"] = [sureness[j][i] for j in range(len(sureness))]

In [81]:
features = [f'f{i}' for i in range(1000)]
features += [f'img{i}' for i in range(len(train_img_emb[0]))]
features += [f'text{i}' for i in range(len(train_text_emb[0]))]
features.append('description')

target = 'label'

In [82]:
X_tr, X_val, y_tr, y_val = train_test_split(train_df[features], train_df[target], test_size=0.3, shuffle=True, stratify=train_df[target])

In [84]:
catboost_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,                  # Deeper, but heavily regularized
    l2_leaf_reg=20,
    rsm=0.2,                  # Only look at 600 features per split!
    random_strength=5.0,      # Heavy penalty for "too perfect" splits
    bagging_temperature=2.0,  # Aggressive row subsampling
    min_data_in_leaf=50,      # Prevents tiny leaves fitting to noise
    early_stopping_rounds=100,
    eval_metric='AUC',
    verbose=200,
    text_features=['description']
)


catboost_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

pred = catboost_model.predict(X_val)
print("Точность на валидационной выборке:", accuracy_score(pred, y_val))

pred = catboost_model.predict_proba(X_val).tolist()
print("ROC AUC на валидационной выборке:", roc_auc_score(y_val, [i[1] for i in pred]))

0:	test: 0.6430349	best: 0.6430349 (0)	total: 53.2ms	remaining: 2m 39s
200:	test: 0.9087289	best: 0.9087289 (200)	total: 11.7s	remaining: 2m 42s
400:	test: 0.9214753	best: 0.9214753 (400)	total: 23.2s	remaining: 2m 30s
600:	test: 0.9311804	best: 0.9311810 (599)	total: 35s	remaining: 2m 19s
800:	test: 0.9353356	best: 0.9353356 (800)	total: 46.9s	remaining: 2m 8s
1000:	test: 0.9377461	best: 0.9377461 (1000)	total: 59s	remaining: 1m 57s
1200:	test: 0.9395385	best: 0.9395385 (1200)	total: 1m 12s	remaining: 1m 47s
1400:	test: 0.9405183	best: 0.9405183 (1400)	total: 1m 24s	remaining: 1m 36s
1600:	test: 0.9414896	best: 0.9414896 (1600)	total: 1m 36s	remaining: 1m 24s
1800:	test: 0.9422667	best: 0.9422742 (1798)	total: 1m 48s	remaining: 1m 12s
2000:	test: 0.9426713	best: 0.9426831 (1990)	total: 2m 1s	remaining: 1m
2200:	test: 0.9434292	best: 0.9434501 (2195)	total: 2m 14s	remaining: 48.8s
2400:	test: 0.9438592	best: 0.9438689 (2378)	total: 2m 28s	remaining: 37s
2600:	test: 0.9443302	best: 0.94

In [86]:
catboost_model.save_model('catboost_model.cbm')

In [ ]:
ts_images_ds = ImageDataset(IMAGE_FOLDER, test_df)

In [ ]:
images_ts = DataLoader(ts_images_ds, batch_size=batch_size, shuffle=False)

In [ ]:
sureness_ts = []

for images_ in tqdm(images_ts):
    with torch.no_grad():
        sure = res_net(images_.to(DEVICE))

    sureness_ts.extend(sure.to('cpu').numpy().tolist())

100%|██████████| 264/264 [01:20<00:00,  3.28it/s]


In [21]:
for i in range(1000):
    test_df[f"f{i}"] = [sureness_ts[j][i] for j in range(len(sureness_ts))]

for i in range(len(train_text_emb[-1])):
    test_df[f"text{i}"] = [test_text_emb[j][i] for j in range(len(test_text_emb))]

for i in range(len(train_img_emb[-1])):
    test_df[f"img{i}"] = [test_img_emb[j][i] for j in range(len(test_img_emb))]

In [87]:
pred_ts = catboost_model.predict_proba(test_df[features])

In [88]:
test_df['y_pred'] = [i[1] for i in pred_ts.tolist()]

In [89]:
(test_df[["id", 'y_pred']]).to_csv("submission_with_resnet.csv", index=False)